### Build Constructor Standings View

In [0]:
CREATE OR REPLACE VIEW formula1.gold.v_constructor_standings
AS (
WITH constructor_session_summary AS(
    SELECT 
        r.season,
        r.constructor_id,
        c.constructor_name,
        c.nationality,
        COUNT(*) AS race_starts,
        SUM(r.points) AS total_points,
        COUNT_IF(r.is_win) AS number_of_wins,
        COUNT_IF(r.is_podium) AS number_of_podiums
    FROM formula1.gold.fact_session_results AS r
    JOIN formula1.gold.dim_constructors AS c
    on r.constructor_id = c.constructor_id
    GROUP BY
        r.season,
        r.constructor_id,
        c.constructor_name,
        c.nationality
)

SELECT
    season,
    constructor_id,
    constructor_name,
    nationality,
    RANK() OVER(PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC, number_of_podiums DESC) AS standings,
    race_starts,
    total_points,
    number_of_wins,
    number_of_podiums
FROM constructor_session_summary
)